### Report purpose

This notebook provides the first model benchmark on the cleaned fare dataset. The goal is not exhaustive optimization yet; it is to compare a small set of leakage-safe regression models under the same evaluation design.

The workflow is intentionally simple: load the prepared table, split train and test chronologically, sample a manageable subset for faster iteration, and compare all candidate models with the same target and metrics.


In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


### Load the modeling dataset

This notebook starts from the cleaned output created in `02_cleaninig.ipynb`.


In [2]:
cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())
data_path = project_root / "data" / "procesed" / "taxi_2019_modeling_ready.parquet"

df = pd.read_parquet(data_path)
print("dataset shape:", df.shape)
df.head()

dataset shape: (11792502, 28)


,fare_amount,pickup_month_num,trip_distance,log_trip_distance,trip_distance_sq,distance_x_rush,distance_x_weekend,passenger_count,passenger_count_missing,pickup_weekday,...,PULocationID,DOLocationID,pickup_borough,dropoff_borough,pickup_service_zone,dropoff_service_zone,is_airport_pickup,is_airport_dropoff,same_borough_trip,manhattan_trip
0,4.5,1,0.52,0.418710,0.2704,0.52,0.0,1.0,0,2,...,164,234,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
1,7.0,1,1.15,0.765468,1.3225,1.15,0.0,1.0,0,2,...,100,230,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
2,10.5,1,2.44,1.235471,5.9536,0.00,0.0,1.0,0,0,...,140,162,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
3,5.5,1,1.20,0.788457,1.4400,0.00,0.0,1.0,0,2,...,151,239,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
4,20.0,1,4.60,1.722767,21.1600,4.60,0.0,1.0,0,3,...,140,260,Manhattan,Queens,Yellow Zone,Boro Zone,0,0,0,1


### Create a chronological train-test split

Training uses January through October and testing uses November through December. This is closer to a realistic forecasting workflow than a random split because the model is evaluated on later trips it has not seen before.


In [3]:
target_col = "fare_amount"
split_col = "pickup_month_num"

train_mask = df[split_col] <= 10
test_mask = df[split_col] >= 11

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

train shape: (9839239, 28)
test shape: (1953263, 28)


### Sample for faster comparison

The full training table is large, so this stage uses fixed random samples to keep experimentation fast and reproducible. Once a strong candidate is identified, the later tuning notebook trains more seriously.


In [4]:
RANDOM_STATE = 42
TRAIN_SAMPLE = 300_000
TEST_SAMPLE = 120_000

train_df = train_df.sample(n=min(TRAIN_SAMPLE, len(train_df)), random_state=RANDOM_STATE)
test_df = test_df.sample(n=min(TEST_SAMPLE, len(test_df)), random_state=RANDOM_STATE)

print("sampled train shape:", train_df.shape)
print("sampled test shape:", test_df.shape)

sampled train shape: (300000, 28)
sampled test shape: (120000, 28)


### Define the feature set

The target is `fare_amount`. The raw split month is excluded from training because that information is already represented through cyclical calendar features in the cleaned dataset.


In [5]:
feature_cols = [c for c in df.columns if c not in [target_col, split_col]]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

categorical_cols = [
    "pickup_borough",
    "dropoff_borough",
    "pickup_service_zone",
    "dropoff_service_zone",
]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("numeric columns:", len(numeric_cols))
print("categorical columns:", len(categorical_cols))

numeric columns: 22
categorical columns: 4


### Build preprocessing pipelines

Linear and tree-based models do not need identical preprocessing, so the notebook creates separate preprocessing blocks. This gives each algorithm a fairer input representation while keeping the evaluation framework consistent.


In [6]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_cols,
        ),
    ]
)


### Compare baseline and candidate models

The model lineup includes a naive median baseline, a regularized linear model, and two nonlinear tree-based models. This progression makes it easier to judge whether additional model complexity is actually earning better predictive performance.


In [7]:
models = {
    "dummy_median": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "ridge": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]),
    "random_forest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=250,
            max_depth=18,
            min_samples_leaf=5,
            n_jobs=1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "gradient_boosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", GradientBoostingRegressor(
            learning_rate=0.08,
            max_depth=6,
            n_estimators=250,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        )),
    ]),
}


### Evaluate on the held-out period

All models are compared on the same unseen period using MAE, RMSE, and R2. These metrics provide a balanced view of average error, sensitivity to larger misses, and overall explanatory power.


In [9]:
results = []

for model_name, pipeline in models.items():
    start = time.time()
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    elapsed = time.time() - start

    results.append(
        {
            "model": model_name,
            "mae": mean_absolute_error(y_test, preds),
            "rmse": root_mean_squared_error(y_test, preds),
            "r2": r2_score(y_test, preds),
            "fit_seconds": elapsed,
        }
    )

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df

,model,mae,rmse,r2,fit_seconds
0,random_forest,1.562832,3.735854,0.897168,424.515320
1,gradient_boosting,1.576290,3.861445,0.890138,372.919681
2,ridge,1.831183,4.027333,0.880495,1.020590
3,dummy_median,6.554074,12.212376,-0.098879,0.900944


### Chapter summary

The best-performing model from this notebook becomes the main candidate for deeper hyperparameter tuning and more detailed error analysis in the next stage.
